<a href="https://colab.research.google.com/github/nykim131/aice/blob/main/sample_text_category.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as ps

train_ds2=ps.read_csv('/content/sample_data/02_train.csv')
test_ds2=ps.read_csv('/content/sample_data/02_test_x.csv')

In [14]:
train_ds2.info()
train_ds2.describe()
train_ds2.isnull().sum()
train_ds2['카테고리'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5931 entries, 0 to 5930
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   상품명     5931 non-null   object
 1   카테고리    5931 non-null   object
dtypes: object(2)
memory usage: 92.8+ KB


,count
카테고리,
햄/소시지(통조림/병제외),1220
건식품,1184
농산물통조림/병,850
해조류,707
수산물통조림/병,609
건포류,576
조미육가공,503
축산물통조림/병,282


In [15]:
train_ds2['상품명'] = train_ds2['상품명'].str.replace('[^가-힣a-zA-Z0-9& ]','', regex=True)
train_ds2['상품명'] = train_ds2['상품명'].str.strip()

In [19]:
train_ds2.drop_duplicates(subset='상품명', inplace=True)

In [17]:
test_ds2['상품명'] = test_ds2['상품명'].str.replace('[^가-힣a-zA-Z0-9& ]','', regex=True)
test_ds2['상품명'] = test_ds2['상품명'].str.strip()

In [20]:
train_ds2.head(50)

,상품명,카테고리
0,존쿡 델리미트 이탈리안 살라미 오리지널&치즈 40g,햄/소시지(통조림/병제외)
1,머거본 꿀땅콩 70g,건식품
2,해왕 국물용대멸치 150g,건포류
3,토농이 씨앗믹스 250g,건식품
4,동원 스위트콘 198g,농산물통조림/병
5,프레시안 더The 건강한 햄 540g,햄/소시지(통조림/병제외)
6,해맑음 국물용멸치 450g,건포류
7,사회적협동조합제주이어도지역자활센터 제주톡말린감귤 20g,건식품
8,프레시안 토종김참기름 30g,해조류
9,육공방 비엔나 치즈 260g,햄/소시지(통조림/병제외)


In [21]:
from sklearn.preprocessing import LabelEncoder

le2 = LabelEncoder()

train_ds2['카테고리'] = le2.fit_transform(train_ds2['카테고리'])

In [22]:
train_ds2['카테고리']

,카테고리
0,7
1,0
2,1
3,0
4,2
...,...
5925,0
5926,2
5927,7
5928,6


In [23]:
from sklearn.model_selection import train_test_split
x_train_2, x_val_2, y_train_2, y_val_2 = train_test_split(train_ds2['상품명'], train_ds2['카테고리'] , test_size=0.2, stratify=train_ds2['카테고리'], random_state=41)

In [24]:
x_train_2.shape, x_val_2.shape, y_train_2.shape, y_val_2.shape

((4529,), (1133,), (4529,), (1133,))

In [25]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(x_train_2)

max_words = max(tokenizer.index_word) +1
print(max_words)


5071


In [26]:
x_train_seq = tokenizer.texts_to_sequences(x_train_2)
x_val_seq = tokenizer.texts_to_sequences(x_val_2)
x_test_seq = tokenizer.texts_to_sequences(test_ds2['상품명'])

In [28]:
print(max(len(line) for line in x_train_seq))
print(max(len(line) for line in x_val_seq))
print(max(len(line) for line in x_test_seq))

max_len = max(len(line) for line in x_train_seq)

17
14
11


In [29]:
x_train_pad = pad_sequences(x_train_seq, maxlen=max_len)
x_val_pad = pad_sequences(x_val_seq, maxlen=max_len)
x_test_pad = pad_sequences(x_test_seq, maxlen=max_len)

In [30]:
print(x_train_pad[:1])
print(x_train_pad.shape, x_val_pad.shape)

[[   0    0    0    0    0    0    0    0    0    0    0 2166  714 2167
    46  147   13]]
(4529, 17) (1133, 17)


In [31]:
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPool2D
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, SimpleRNN, GRU
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model2 = Sequential()

model2.add(Embedding(max_words, 32, input_length=max_len)) # 단어를 의미있는 32 차원으로 Vector 변경(Embedding)

# LSTM 모델
model2.add(LSTM(16, return_sequences=True))
model2.add(LSTM(16, return_sequences=True))
model2.add(Flatten())
model2.add(Dense(128, activation='swish'))
model2.add(Dense(32, activation='swish'))
model2.add(Dense(8, activation='softmax')) # 최종 8가지로 분류

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:

model2.compile(loss = 'sparse_categorical_crossentropy',
              optimizer = 'adam',
              metrics = ['accuracy'])
model2.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:

es = EarlyStopping(monitor='val_loss', patience=3, verbose=1)
checkpoint_path = 'tmp_checkpoint.h5'
cp = ModelCheckpoint(checkpoint_path, monitor='val_loss', verbose=1, save_best_only=True)

history = model2.fit(x_train_pad, y_train_2, epochs=50, batch_size=512,
                      validation_split=0.2, verbose =1, callbacks=[es, cp])


Epoch 1/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.1758 - loss: 2.0664
Epoch 1: val_loss improved from None to 2.01289, saving model to tmp_checkpoint.h5



Epoch 1: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.1885 - loss: 2.0485 - val_accuracy: 0.2009 - val_loss: 2.0129
Epoch 2/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2008 - loss: 2.0061
Epoch 2: val_loss improved from 2.01289 to 2.00051, saving model to tmp_checkpoint.h5



Epoch 2: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.2012 - loss: 2.0047 - val_accuracy: 0.2075 - val_loss: 2.0005
Epoch 3/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2068 - loss: 1.9951
Epoch 3: val_loss improved from 2.00051 to 1.99447, saving model to tmp_checkpoint.h5



Epoch 3: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2040 - loss: 1.9954 - val_accuracy: 0.2009 - val_loss: 1.9945
Epoch 4/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.2059 - loss: 1.9896
Epoch 4: val_loss did not improve from 1.99447
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.1974 - loss: 1.9932 - val_accuracy: 0.2009 - val_loss: 1.9947
Epoch 5/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2012 - loss: 1.9996
Epoch 5: val_loss improved from 1.99447 to 1.99215, saving model to tmp_checkpoint.h5



Epoch 5: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.2059 - loss: 1.9918 - val_accuracy: 0.2075 - val_loss: 1.9921
Epoch 6/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2082 - loss: 1.9861
Epoch 6: val_loss improved from 1.99215 to 1.98991, saving model to tmp_checkpoint.h5


...


Epoch 15: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5377 - loss: 1.1740 - val_accuracy: 0.5177 - val_loss: 1.3770
Epoch 16/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6052 - loss: 1.0778
Epoch 16: val_loss improved from 1.37701 to 1.33906, saving model to tmp_checkpoint.h5



Epoch 16: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6047 - loss: 1.0267 - val_accuracy: 0.4923 - val_loss: 1.3391
Epoch 17/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6235 - loss: 0.9611
Epoch 17: val_loss improved from 1.33906 to 1.07834, saving model to tmp_checkpoint.h5



Epoch 17: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6542 - loss: 0.9068 - val_accuracy: 0.6049 - val_loss: 1.0783
Epoch 18/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7210 - loss: 0.7395
Epoch 18: val_loss improved from 1.07834 to 1.04103, saving model to tmp_checkpoint.h5



Epoch 18: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.7320 - loss: 0.7434 - val_accuracy: 0.6887 - val_loss: 1.0410
Epoch 19/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7716 - loss: 0.6464
Epoch 19: val_loss did not improve from 1.04103
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7704 - loss: 0.6324 - val_accuracy: 0.6821 - val_loss: 1.0723
Epoch 20/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8196 - loss: 0.5579
Epoch 20: val_loss improved from 1.04103 to 0.98278, saving model to tmp_checkpoint.h5



Epoch 20: finished saving model to tmp_checkpoint.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.8405 - loss: 0.5162 - val_accuracy: 0.7230 - val_loss: 0.9828
Epoch 21/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8604 - loss: 0.4290
Epoch 21: val_loss did not improve from 0.98278
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8625 - loss: 0.4243 - val_accuracy: 0.6667 - val_loss: 1.1840
Epoch 22/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8535 - loss: 0.4337
Epoch 22: val_loss did not improve from 0.98278
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.8777 - loss: 0.3924 - val_accuracy: 0.7042 - val_loss: 1.1038
Epoch 23/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9020 - loss: 0.3337
Epoch 23: val_loss did not improve from 0.98278
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - accuracy: 0.9103 - loss: 0.3146 - val_accuracy: 0.7130 - val_loss: 1.1084
Epoch 23: early stopping


In [34]:
model2.evaluate(x_val_pad, y_val_2)

36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7193 - loss: 0.9782


[0.9782332181930542, 0.7193292379379272]

In [37]:
import numpy as np

pred = np.argmax(model2.predict(x_test_pad))

result = np.argmax(model2.predict(x_test_pad), axis=1)
result = le2.inverse_transform(result)

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [38]:
result

array(['농산물통조림/병', '건포류', '농산물통조림/병', '햄/소시지(통조림/병제외)', '건식품', '건포류',
       '건식품', '조미육가공', '건포류', '건식품', '건식품', '건포류', '해조류',
       '햄/소시지(통조림/병제외)', '햄/소시지(통조림/병제외)', '건식품', '햄/소시지(통조림/병제외)', '건포류',
       '건포류', '햄/소시지(통조림/병제외)', '축산물통조림/병', '햄/소시지(통조림/병제외)', '수산물통조림/병',
       '햄/소시지(통조림/병제외)', '해조류', '햄/소시지(통조림/병제외)', '건식품', '농산물통조림/병',
       '축산물통조림/병', '건포류', '건식품', '해조류', '농산물통조림/병', '농산물통조림/병',
       '농산물통조림/병', '건식품', '건포류', '건식품', '수산물통조림/병', '건포류', '건포류', '건포류',
       '햄/소시지(통조림/병제외)', '농산물통조림/병', '건식품', '건식품', '농산물통조림/병', '농산물통조림/병',
       '농산물통조림/병', '축산물통조림/병', '농산물통조림/병', '건식품', '조미육가공',
       '햄/소시지(통조림/병제외)', '농산물통조림/병', '조미육가공', '해조류', '조미육가공', '건식품',
       '조미육가공', '농산물통조림/병', '건식품', '해조류', '햄/소시지(통조림/병제외)', '수산물통조림/병',
       '햄/소시지(통조림/병제외)', '건포류', '건식품', '조미육가공', '건포류', '건식품', '수산물통조림/병',
       '농산물통조림/병', '햄/소시지(통조림/병제외)', '수산물통조림/병', '햄/소시지(통조림/병제외)', '건포류',
       '건식품', '농산물통조림/병', '축산물통조림/병', '해조류', '건식품', '건식품', '수산물통조림/병',
       '농산물통조림/병', '수산물

In [39]:
test_ds2=ps.read_csv('/content/sample_data/02_test_x.csv')

In [40]:
test_ds2['카테고리'] = result

In [41]:
test_ds2

,상품명,카테고리
0,paste it 페이스트 잇 검은깨 175g,농산물통조림/병
1,맛군밤 180g,건포류
2,Rio Santo 케이퍼 110g,농산물통조림/병
3,청정원 건강생각 비엔나프라임 450g,햄/소시지(통조림/병제외)
4,구운캐슈너트 250 g,건식품
...,...,...
512,건포도 500g,건식품
513,건강한참치저나트륨,건포류
514,그릭슈바인 육즙가득 부어스트 바질 710g,햄/소시지(통조림/병제외)
515,땅콩버터오다리 110g,건포류


In [43]:
import joblib
joblib.dump(model2, '010_123123.h5')

['010_123123.h5']

In [44]:
loaded_model = joblib.load('010_123123.h5')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 15 variables whereas the saved optimizer has 28 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [47]:
loaded_model.evaluate(x_val_pad, y_val_2)

36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7193 - loss: 0.9782


[0.9782332181930542, 0.7193292379379272]